In [ ]:
#%pip install msoffcrypto-tool
#%pip install pywin32

In [ ]:
#%pip install python-calamine

In [ ]:
import msoffcrypto
import pandas as pd
import io
import openpyxl

In [ ]:
pd.set_option('display.max_rows', None)

In [ ]:
# 1. กำหนดชื่อไฟล์และรหัสผ่าน
file_path = r'H:\My Drive\การเงิน\ยอดขาย\2569\ยอดขายรวมทุกสาขาBplus2569.xlsx'
password = '18651865'

In [ ]:
data = []
month_names = ['ม.ค.69', 'ก.พ.69', 'มี.ค.69', 'เม.ย.69', 'พ.ค.69', 'มิ.ย.69', 
               'ก.ค.69', 'ส.ค.69', 'ก.ย.69', 'ต.ค.69', 'พ.ย.69', 'ธ.ค.69']
# 2. สร้างออบเจ็กต์สำหรับจัดการไฟล์ที่ล็อก
temp_file = io.BytesIO()    

with open(file_path, 'rb') as f:
    office_file = msoffcrypto.OfficeFile(f)
    
    # ใส่รหัสผ่านเพื่อปลดล็อก
    office_file.load_key(password=password)
    
    # บันทึกไฟล์ที่ปลดล็อกแล้วลงในหน่วยความจำ (temp_file)
    office_file.decrypt(temp_file)

In [ ]:
cols = 'A:H,T'
# 3. ใช้ Pandas อ่านไฟล์จากหน่วยความจำ
for i in month_names:
    df = pd.read_excel(temp_file , sheet_name= i ,usecols=cols, header= 4 ,engine= 'openpyxl')
    df = df.iloc[0:35,:]
    # แปลงข้อมูลเป็นวันที่
    df['DATE'] = pd.to_datetime(df['Unnamed: 0'], dayfirst=True ,errors='coerce')
    #ลบค่าว่าง
    df.dropna(subset='DATE',inplace=True)
    #กรอกวันที่ที่มากกว่า 2020
    df = df[df['DATE'].dt.year > 2020]

    df['DATE'] = df['DATE'].dt.date

    df.drop(columns=['Unnamed: 0'] , inplace=True)
    
    df.rename(columns={'Unnamed: 19': 'WH'}, inplace=True)

    data.append(df)

full = pd.concat(data , ignore_index=True)
# ลบซ้ำ
full.drop_duplicates(inplace=True)

full.insert(0 ,'DATE1',full['DATE'])

full.drop(columns='DATE',inplace=True)

full.rename(columns={'DATE1':'DATE'},inplace=True)



In [ ]:
full